In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install transformers
!pip install sentencepiece
!pip install datasets
!pip install langdetect # Only for cleaning dataset

## Local Inference on GPU 
Model page: https://huggingface.co/vesteinn/ScandiBERT

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/vesteinn/ScandiBERT)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("kaggle-scandibert")

import torch
import math
import re
import langdetect # ONLY for cleaning dataset
from huggingface_hub import hf_hub_download
from transformers import XLMRobertaTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset, concatenate_datasets
from langdetect import detect

model_name = "vesteinn/ScandiBERT" # Considering no-faroese variant for slight tokenization difference
labels = ["Norwegian", "Non-Scandinavian"]
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

# Hard-coded because tokenizer.json is broken
vocab_path = hf_hub_download(repo_id=model_name, filename="sentencepiece.bpe.model")
tokenizer = XLMRobertaTokenizer(vocab_file=vocab_path)

SENTENCES_PER_CLASS = 3000

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels = len(labels),
    id2label = id2label,  
    label2id = label2id,
    ignore_mismatched_sizes = True,
    problem_type = "multi_label_classification"
)

with torch.no_grad(): #Delete if no effect
    model.classifier.dense.weight.data.normal_(mean=0.0, std=0.001)
    model.classifier.dense.bias.data.zero_()
    model.classifier.out_proj.weight.data.normal_(mean=0.0, std=0.001)
    model.classifier.out_proj.bias.data.zero_()

def get_opus_sentences(config, lang_code, num_sentences=SENTENCES_PER_CLASS):
    print(f"Downloading {lang_code} from {config}...")
    ds = load_dataset("Helsinki-NLP/opus-100", config, split="train")
    ds = ds.filter(lambda sentence: is_dataset_clean(sentence["translation"][lang_code]))
    ds = ds.shuffle(seed=42).select(range(num_sentences))
    return [item[lang_code] for item in ds["translation"]]

def is_dataset_clean(text, instance_threshold=2, min_length=4, max_length=20):
    bad_char_regex = r"[@#$%^&*_=+\[\]{};':\"<>/`~\\|]"
    bad_format_regex = r'[A-Z][a-z]+[A-Z]|%\s*\d|https?://'
    if not isinstance(text, str):
        return False
    if len(text.strip()) < max_length or len(text.split()) < min_length:
        return False
    if len(re.findall(bad_char_regex, text)) >= instance_threshold:
        return False
    if re.search(bad_format_regex, text):
        return False
    return True

def one_lang_text_only(text, lang_codes):
    try:
        return detect(text) in lang_codes
    except:
        return False
        

def sent_based_int(percent, sent_const=SENTENCES_PER_CLASS): # Get an integer that's a % of SENTENCES_PER_CLASS
    return int(math.ceil(sent_const * percent))

sample_norwegian_all = get_opus_sentences("en-nb", "nb", sent_based_int(0.75)) + get_opus_sentences("en-nn", "nn", sent_based_int(0.25))
sample_norwegian_all = [t for t in sample_norwegian_all if one_lang_text_only(t, ["no", "nb", "nn"])]
nonscandi_configs = [("en-es", "en"), ("en-es", "es"), ("de-en", "de"), ("en-nl", "nl"), ("af-en", "af")]
sample_nonscandi_all = []
for config, code in nonscandi_configs:
    sample_nonscandi_all.extend(get_opus_sentences(config, code, sent_based_int(1 / len(nonscandi_configs))))
print(f"Norwegian corpora collected: {len(sample_norwegian_all)} sentences.")
print(f"Non-Scandinavian corpora collected: {len(sample_nonscandi_all)} sentences.")

text = "Dette er en norsk setning."
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
print("\nZero-Shot Classification Probabilities: ")
probs = torch.sigmoid(outputs.logits)
for label, prob in zip(labels, probs[0]): # True "zero-shot", random classification test
    print(f"{label}: {prob.item():.4f}")

In [ ]:
from datasets import Dataset, concatenate_datasets
from sklearn.model_selection import train_test_split

eval_ds_split = 0.15

norwegian_train, norwegian_eval = train_test_split(
    sample_norwegian_all, test_size=eval_ds_split, random_state=42
)
nonscandi_train, nonscandi_eval = train_test_split(
    sample_nonscandi_all, test_size=eval_ds_split, random_state=42
)

dataset_norwegian_train = Dataset.from_dict({"text": norwegian_train, "labels": [[1.0, 0.0]] * len(norwegian_train)})
dataset_nonscandi_train = Dataset.from_dict({"text": nonscandi_train, "labels": [[0.0, 1.0]] * len(nonscandi_train)})
train_dataset = concatenate_datasets([dataset_norwegian_train, dataset_nonscandi_train]).shuffle(seed=42)

dataset_norwegian_eval = Dataset.from_dict({"text": norwegian_eval, "labels": [[1.0, 0.0]] * len(norwegian_eval)})
dataset_nonscandi_eval = Dataset.from_dict({"text": nonscandi_eval, "labels": [[0.0, 1.0]] * len(nonscandi_eval)})
eval_dataset = concatenate_datasets([dataset_norwegian_eval, dataset_nonscandi_eval]).shuffle(seed=42)

print(train_dataset[:20])

def tokenize_dataset(select_dataset):
    return tokenizer(select_dataset["text"], truncation=True, max_length=512)

print(f"{len(train_dataset)} Training samples | {len(eval_dataset)} Eval samples")

train_dataset = train_dataset.map(tokenize_dataset, batched=True)
eval_dataset = eval_dataset.map(tokenize_dataset, batched=True)
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
import re
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir = "./scandiprob_checkpoints",
    learning_rate = 2e-5,                    # 0.00002 learning rate
    per_device_train_batch_size = 8,         # Adjust to 4 if Kaggle throws an OutOfMemory (OOM) error
    gradient_accumulation_steps =1,
    num_train_epochs = 10,                   
    weight_decay = 0.01,
    fp16 = True,
    warmup_steps = sent_based_int(0.01),
    report_to = "none",
    dataloader_pin_memory = False,
    logging_steps = sent_based_int(0.1),
    eval_strategy = "epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

def nonscandi_penalty(text): # Regex for negative class
    if not text:
        return 0.0

    scandi_keyboard = r"[a-zA-ZæøåÆØÅäöÄÖéÉ0-9 !@#$%^&*()\-_=+\[\]{};':\",.<>?/`~\\|]"
    scandi_matches = re.findall(scandi_keyboard, text)
    nonscandi_percent = (1 - (len(scandi_matches) / len(text)))
    return nonscandi_percent

print("Training...")
trainer.train()
trainer.save_model("./scandiprob_current")
print("Training complete and model saved!")

test_text = "Dette er en norsk setning."
test_inputs = tokenizer(test_text, return_tensors="pt")
test_inputs = {k: v.to(model.device) for k, v in test_inputs.items()}
model.eval()

with torch.no_grad():
    outputs = model(**test_inputs)

print("\n")
raw_probs = torch.sigmoid(outputs.logits)[0]

nonscandi_ratio = nonscandi_penalty(test_text)


final_probs = []

for i, label in enumerate(labels):
    prob = raw_probs[i].item()
    
    if label == "Norwegian":
        adjusted = prob * (1.0 - nonscandi_ratio) 
    elif label == "Non-Scandinavian": # Scale up/down by % of non-Scandi chars
        adjusted = prob + ((1.0 - prob) * nonscandi_ratio)
        
    final_probs.append(adjusted)

skew_effect = (raw_probs != final_probs)
if skew_effect:
    print("\nPre-Adjustment Probabilities: ")
    for label, prob in zip(labels, raw_probs): # Print pre-regex-skew results!
        print(f"{label}: {prob:.4f}")
print("\nPost-Adjustment Probabilities: " if skew_effect else "\nPost-Training Probabilities")
for label, prob in zip(labels, final_probs): # Print post-regex-skew results!
    print(f"{label}: {prob:.4f}")